In [12]:
# ========================
# RandomForest & LightGBM
# ========================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (balanced_accuracy_score, roc_auc_score,
                             precision_recall_curve, classification_report, auc)

from sklearn.metrics import balanced_accuracy_score, roc_auc_score, precision_recall_curve, auc

import joblib

In [13]:
# ======================
#  DATA LOADING & PREPROCESSING
# ======================

df = pd.read_csv('data.csv')

# Efficient cross-feature and binarized flag generation using preallocation
cross_features = {}
binarized_flags = {}

for i in range(1, 31):
    for j in range(i + 1, 31):
        cross_features[f'V{i}_x_V{j}'] = df[f'V{i}'] * df[f'V{j}']
    binarized_flags[f'V{i}_gt_mean'] = (df[f'V{i}'] > df[f'V{i}'].mean()).astype(int)

# Concatenate all new features at once
df = pd.concat([df, pd.DataFrame(cross_features), pd.DataFrame(binarized_flags)], axis=1)
df = df.copy()  # Defragment to avoid future warnings

# Target and features
y = df['OBJ']
X = df.drop(columns=['OBJ'])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
# ======================
#  CLASSIFIERS, HYPERPARAMETERS, GRID & TRAINING
# ======================

# Define classifiers
# Random Forest
rf_clf = RandomForestClassifier(
    random_state=42, 
    class_weight='balanced'
)
# LightGBM
lgb_clf = lgb.LGBMClassifier(
    random_state=42,
    class_weight='balanced',
    force_col_wise=True,
    verbose=-1
)

# Define parameters
# Random Forest
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10],
    'min_samples_split': [2, 5],
}

# LightGBM
param_grid_lgb = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10],
    'learning_rate': [0.05, 0.1],
    'min_child_samples': [20, 50],
    'num_leaves': [31, 50]
}

# Grid search for Random Forest
rf_grid = GridSearchCV(
    rf_clf, param_grid_rf, scoring='balanced_accuracy', cv=5, n_jobs=-1
)
rf_grid.fit(X_train_scaled, y_train)

# Grid search for LGBM
lgb_grid = GridSearchCV(
    lgb_clf, param_grid_lgb, scoring='balanced_accuracy', cv=5, n_jobs=-1
)
lgb_grid.fit(X_train_scaled, y_train)

GridSearchCV(cv=5,
             estimator=LGBMClassifier(class_weight='balanced',
                                      force_col_wise=True, random_state=42,
                                      verbose=-1),
             n_jobs=-1,
             param_grid={'learning_rate': [0.05, 0.1], 'max_depth': [5, 10],
                         'min_child_samples': [20, 50],
                         'n_estimators': [100, 200], 'num_leaves': [31, 50]},
             scoring='balanced_accuracy')

In [15]:
# ======================
#  RESULTS ANALYSIS
# ======================

def evaluate_model(name, model, X_test, y_test):
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)

    bal_acc = balanced_accuracy_score(y_test, preds)
    roc_auc = roc_auc_score(y_test, probs)
    precision, recall, _ = precision_recall_curve(y_test, probs)
    pr_auc = auc(recall, precision)

    print(f"--- {name} ---")
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"PR-AUC: {pr_auc:.4f}")
    print(classification_report(y_test, preds))
    print()

In [16]:
# ======================
#  MODEL SELECTION
# ======================
# Mapping
y_test_mapped = y_test.map({'NO': 0, 'SI': 1})

# Extract best models
rf_best_model = rf_grid.best_estimator_
lgb_best_model = lgb_grid.best_estimator_

# Predict probabilities
rf_probs = rf_best_model.predict_proba(X_test_scaled)[:, 1]
lgb_probs = lgb_best_model.predict_proba(X_test_scaled)[:, 1]

# Calculate evaluation metrics
rf_preds = (rf_probs >= 0.5).astype(int)
lgb_preds = (lgb_probs >= 0.5).astype(int)

rf_bal_acc = balanced_accuracy_score(y_test_mapped, rf_preds)
lgb_bal_acc = balanced_accuracy_score(y_test_mapped, lgb_preds)

rf_precision, rf_recall, _ = precision_recall_curve(y_test_mapped, rf_probs)
lgb_precision, lgb_recall, _ = precision_recall_curve(y_test_mapped, lgb_probs)

rf_pr_auc = auc(rf_recall, rf_precision)
lgb_pr_auc = auc(lgb_recall, lgb_precision)

rf_roc_auc = roc_auc_score(y_test_mapped, rf_probs)
lgb_roc_auc = roc_auc_score(y_test_mapped, lgb_probs)

# Summary of all metrics
print("=== Final Model Comparison ===")
print(f"Random Forest -> Balanced Acc: {rf_bal_acc:.4f}, PR-AUC: {rf_pr_auc:.4f}, ROC-AUC: {rf_roc_auc:.4f}")
print(f"LightGBM      -> Balanced Acc: {lgb_bal_acc:.4f}, PR-AUC: {lgb_pr_auc:.4f}, ROC-AUC: {lgb_roc_auc:.4f}")

# Select best model (majority win among 3 metrics)
rf_wins = sum([
    rf_bal_acc > lgb_bal_acc,
    rf_pr_auc > lgb_pr_auc,
    rf_roc_auc > lgb_roc_auc
])

if rf_wins >= 2:
    final_model = rf_best_model
    final_model_name = "Random Forest"
else:
    final_model = lgb_best_model
    final_model_name = "LightGBM"

# Save selected final model
print(f"Final Selected Model: {final_model_name}")
joblib.dump(final_model, f"final_{final_model_name.replace(' ', '_').lower()}_model.pkl")


=== Final Model Comparison ===
Random Forest -> Balanced Acc: 0.5953, PR-AUC: 0.3664, ROC-AUC: 0.6194
LightGBM      -> Balanced Acc: 0.5897, PR-AUC: 0.3620, ROC-AUC: 0.6264
Final Selected Model: Random Forest


['final_random_forest_model.pkl']